# Melanoma Research Assistant — Data Ingestion & Embedding Pipeline

This notebook:
1. Fetches melanoma-related abstracts from **PubMed** via the NCBI Entrez API
2. Cleans and chunks the text with LangChain
3. Generates embeddings with **OpenAI `text-embedding-3-small`**
4. Stores vectors + metadata (PMID, title, year, journal) in **Pinecone**

> Run cells top to bottom. You'll need an **NCBI email**, an **OpenAI API key**, and a **Pinecone API key**.


## 0. Install dependencies

In [ ]:
# !pip install -q biopython langchain langchain-openai langchain-pinecone pinecone-client pandas tqdm python-dotenv

## 1. Configuration

Store secrets as environment variables (e.g. in a `.env` file or your shell) rather than hardcoding them.
`.env` example:
```
OPENAI_API_KEY=sk-...
PINECONE_API_KEY=pcn-...
NCBI_EMAIL=you@example.com
NCBI_API_KEY=            # optional, raises rate limit from 3 to 10 req/sec
```


In [ ]:
import os
import time
import json
import re

from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
NCBI_EMAIL = os.environ.get("NCBI_EMAIL", "your_email@example.com")
NCBI_API_KEY = os.environ.get("NCBI_API_KEY")  # optional

# Pinecone index config
PINECONE_INDEX_NAME = "melanoma-kb"
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536 

# PubMed search scope
PUBMED_QUERY = '("melanoma"[MeSH Terms] OR "melanoma"[Title/Abstract]) AND ("2015"[PDAT] : "3000"[PDAT])'
MAX_RESULTS = 5000 


## 2. Fetch abstracts from PubMed (Entrez API)


In [2]:
from Bio import Entrez, Medline

Entrez.email = NCBI_EMAIL
if NCBI_API_KEY:
    Entrez.api_key = NCBI_API_KEY

def search_pubmed(query: str, max_results: int) -> list[str]:
    """Return a list of PMIDs matching the query."""
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results, sort="relevance")
    record = Entrez.read(handle)
    handle.close()
    return record["IdList"]

pmids = search_pubmed(PUBMED_QUERY, MAX_RESULTS)
print(f"Found {len(pmids)} PMIDs for query: {PUBMED_QUERY}")


Found 5000 PMIDs for query: ("melanoma"[MeSH Terms] OR "melanoma"[Title/Abstract]) AND ("2015"[PDAT] : "3000"[PDAT])


In [3]:
def fetch_records(pmids: list[str], batch_size: int = 200) -> list[dict]:
    """Fetch MEDLINE records for a list of PMIDs, in batches to respect rate limits."""
    all_records = []
    for i in range(0, len(pmids), batch_size):
        batch = pmids[i:i + batch_size]
        handle = Entrez.efetch(db="pubmed", id=batch, rettype="medline", retmode="text")
        records = list(Medline.parse(handle))
        handle.close()
        all_records.extend(records)
        print(f"Fetched {len(all_records)}/{len(pmids)} records...")
        time.sleep(0.4 if NCBI_API_KEY else 1.0)  # be polite to NCBI servers
    return all_records

raw_records = fetch_records(pmids)
print(f"Total records fetched: {len(raw_records)}")


Fetched 200/5000 records...
Fetched 400/5000 records...
Fetched 600/5000 records...
Fetched 800/5000 records...
Fetched 1000/5000 records...
Fetched 1200/5000 records...
Fetched 1400/5000 records...
Fetched 1600/5000 records...
Fetched 1800/5000 records...
Fetched 2000/5000 records...
Fetched 2200/5000 records...
Fetched 2400/5000 records...
Fetched 2600/5000 records...
Fetched 2800/5000 records...
Fetched 3000/5000 records...
Fetched 3200/5000 records...
Fetched 3400/5000 records...
Fetched 3600/5000 records...
Fetched 3800/5000 records...
Fetched 4000/5000 records...
Fetched 4200/5000 records...
Fetched 4400/5000 records...
Fetched 4600/5000 records...
Fetched 4800/5000 records...
Fetched 5000/5000 records...
Total records fetched: 5000


In [4]:
raw_records[0]

{'PMID': '33759772',
 'OWN': 'NLM',
 'STAT': 'MEDLINE',
 'DCOM': '20220120',
 'LR': '20220120',
 'IS': '1558-1977 (Electronic) 0889-8588 (Linking)',
 'VI': '35',
 'IP': '1',
 'DP': '2021 Feb',
 'TI': 'Biology of Melanoma.',
 'PG': '29-56',
 'LID': 'S0889-8588(20)30108-8 [pii] 10.1016/j.hoc.2020.08.010 [doi]',
 'AB': 'Melanoma skin cancer is derived from skin melanocytes and has a high risk of metastatic spread. The era of molecular genetics and next-generation sequencing has uncovered the role of oncogenic BRAFV600E mutations in many melanomas, validated the role of ultraviolet-induced DNA mutations in melanoma formation, and uncovered many of the molecular events that occur during melanoma development. Targeted therapies and immunotherapy have dramatically improved outcomes and provided an increased rate of cure for metastatic melanoma. This article reviews the formation of melanoma, the molecular events involved in melanoma growth and metastasis, and the biology underlying resistance

In [5]:
def record_to_doc(rec: dict) -> dict | None:
    """Extract the fields we care about from a MEDLINE record. Returns None if no abstract."""
    abstract = rec.get("AB")
    if not abstract:
        return None  # skip records with no abstract — nothing to embed
    return {
        "pmid": rec.get("PMID", ""),
        "title": rec.get("TI", "").strip(),
        "abstract": abstract.strip(),
        "journal": rec.get("TA", "Unknown journal"),
        "year": (rec.get("DP", "")[:4] if rec.get("DP") else "Unknown"),
        "authors": rec.get("AU", []),
        "doi": next((aid.split(" ")[0] for aid in rec.get("AID", []) if "[doi]" in aid), ""),
    }

docs = [d for d in (record_to_doc(r) for r in raw_records) if d is not None]
print(f"Usable records with abstracts: {len(docs)} (dropped {len(raw_records) - len(docs)} with no abstract)")


Usable records with abstracts: 4787 (dropped 213 with no abstract)


In [6]:
docs[0]

{'pmid': '33759772',
 'title': 'Biology of Melanoma.',
 'abstract': 'Melanoma skin cancer is derived from skin melanocytes and has a high risk of metastatic spread. The era of molecular genetics and next-generation sequencing has uncovered the role of oncogenic BRAFV600E mutations in many melanomas, validated the role of ultraviolet-induced DNA mutations in melanoma formation, and uncovered many of the molecular events that occur during melanoma development. Targeted therapies and immunotherapy have dramatically improved outcomes and provided an increased rate of cure for metastatic melanoma. This article reviews the formation of melanoma, the molecular events involved in melanoma growth and metastasis, and the biology underlying resistance to melanoma therapies.',
 'journal': 'Hematol Oncol Clin North Am',
 'year': '2021',
 'authors': ['Ostrowski SM', 'Fisher DE'],
 'doi': '10.1016/j.hoc.2020.08.010'}

In [2]:
import pandas as pd

# df = pd.DataFrame(docs)
# os.makedirs("data", exist_ok=True)
# df.to_json("data/pubmed_melanoma_abstracts_raw.jsonl", orient="records", lines=True)
# df.head()

df = pd.read_json("data/pubmed_melanoma_abstracts_raw.jsonl", orient="records", lines=True)
df.head()


,pmid,title,abstract,journal,year,authors,doi
0,33759772,Biology of Melanoma.,Melanoma skin cancer is derived from skin mela...,Hematol Oncol Clin North Am,2021,"[Ostrowski SM, Fisher DE]",10.1016/j.hoc.2020.08.010
1,31366280,Current state of melanoma diagnosis and treatm...,Melanoma is the deadliest form of skin cancer....,Cancer Biol Ther,2019,"[Davis LE, Shalin SC, Tackett AJ]",10.1080/15384047.2019.1640032
2,33052224,BET inhibitor suppresses melanoma progression ...,Background: Bromodomain and extra-terminal dom...,Theranostics,2020,"[Deng G, Zeng F, Su J, Zhao S, Hu R, Zhu W, Hu...",10.7150/thno.47432
3,37395165,Anorectal melanoma.,Anorectal melanoma is an aggressive mucosal me...,J Surg Oncol,2023,"[Fastner S, Hieken TJ, McWilliams RR, Hyngstro...",10.1002/jso.27381
4,27165365,Zebrafish Melanoma.,Melanoma skin cancer is a potentially deadly d...,Adv Exp Med Biol,2016,[Kaufman CK],10.1007/978-3-319-30654-4_19


## 3. Clean text

Strip stray HTML/XML tags, normalize whitespace, and drop near-duplicate/empty abstracts.


In [3]:
def clean_text(text: str) -> str:
    text = re.sub(r"<[^>]+>", " ", text)          # strip HTML/XML tags
    text = re.sub(r"\s+", " ", text).strip()      # collapse whitespace
    return text

df["title"] = df["title"].apply(clean_text)
df["abstract"] = df["abstract"].apply(clean_text)

# drop rows with very short abstracts (likely stubs, not real content)
df = df[df["abstract"].str.len() > 100].reset_index(drop=True)
print(f"Records after cleaning: {len(df)}")


Records after cleaning: 4783


## 4. Chunk text with LangChain

Most abstracts are short enough to be a single chunk, but we still run them through a splitter
so longer abstracts (or future full-text additions) are handled consistently.
Each chunk carries full metadata so citations survive retrieval.


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

lc_documents: list[Document] = []
for _, row in df.iterrows():
    full_text = f"Title: {row['title']}\n\nAbstract: {row['abstract']}"
    chunks = splitter.split_text(full_text)
    for idx, chunk in enumerate(chunks):
        lc_documents.append(
            Document(
                page_content=chunk,
                metadata={
                    "pmid": row["pmid"],
                    "title": row["title"],
                    "journal": row["journal"],
                    "year": row["year"],
                    "doi": row["doi"],
                    "chunk_index": idx,
                    "source": "pubmed",
                },
            )
        )

print(f"Total chunks to embed: {len(lc_documents)}")


Total chunks to embed: 15315


## 5. Initialize Pinecone and create the index (if it doesn't exist)


In [5]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

existing_indexes = [idx["name"] for idx in pc.list_indexes()]

if PINECONE_INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Created index: {PINECONE_INDEX_NAME}")
else:
    print(f"Index already exists: {PINECONE_INDEX_NAME}")

index = pc.Index(PINECONE_INDEX_NAME)


Created index: melanoma-kb


## 6. Embed chunks and upsert into Pinecone

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, openai_api_key=OPENAI_API_KEY)

vectorstore = PineconeVectorStore(index=index, embedding=embeddings, text_key="text")

BATCH_SIZE = 200
for i in range(0, len(lc_documents), BATCH_SIZE):
    batch = lc_documents[i:i + BATCH_SIZE]
    ids = [f"{doc.metadata['pmid']}-{doc.metadata['chunk_index']}" for doc in batch]
    vectorstore.add_documents(documents=batch, ids=ids)
    print(f"Upserted {min(i + BATCH_SIZE, len(lc_documents))}/{len(lc_documents)} chunks")

print("Done embedding and upserting to Pinecone.")


Upserted 200/15315 chunks
Upserted 400/15315 chunks
Upserted 600/15315 chunks
Upserted 800/15315 chunks
Upserted 1000/15315 chunks
Upserted 1200/15315 chunks
Upserted 1400/15315 chunks
Upserted 1600/15315 chunks
Upserted 1800/15315 chunks
Upserted 2000/15315 chunks
Upserted 2200/15315 chunks
Upserted 2400/15315 chunks
Upserted 2600/15315 chunks
Upserted 2800/15315 chunks
Upserted 3000/15315 chunks
Upserted 3200/15315 chunks
Upserted 3400/15315 chunks
Upserted 3600/15315 chunks
Upserted 3800/15315 chunks
Upserted 4000/15315 chunks
Upserted 4200/15315 chunks
Upserted 4400/15315 chunks
Upserted 4600/15315 chunks
Upserted 4800/15315 chunks
Upserted 5000/15315 chunks
Upserted 5200/15315 chunks
Upserted 5400/15315 chunks
Upserted 5600/15315 chunks
Upserted 5800/15315 chunks
Upserted 6000/15315 chunks
Upserted 6200/15315 chunks
Upserted 6400/15315 chunks
Upserted 6600/15315 chunks
Upserted 6800/15315 chunks
Upserted 7000/15315 chunks
Upserted 7200/15315 chunks
Upserted 7400/15315 chunks
Upser

## 7. Sanity check — run a test similarity query

In [7]:
results = vectorstore.similarity_search(
    "What are the treatment options for BRAF-mutant metastatic melanoma?",
    k=10,
)

for r in results:
    print(f"PMID {r.metadata['pmid']} | {r.metadata['title']} ({r.metadata['year']})")
    print(r.page_content[:250], "...\n")


PMID 26743513.0 | Molecular Targeted Therapy Approaches for BRAF Wild-Type Melanoma. (2016.0)
. The treatment of this subset of patients is a challenging problem. In recent years, preclinical and early clinical studies have suggested that inhibitors of mitogen activated protein kinase (MAPK) pathway and parallel signaling networks may have ac ...

PMID 32228358.0 | Systemic Therapy for Melanoma: ASCO Guideline. (2020.0)
. In the unresectable/metastatic setting, ipilimumab plus nivolumab, nivolumab alone, or pembrolizumab alone should be offered to patients with BRAF wild-type cutaneous melanoma, while those three regimens or combination BRAF/MEK inhibitor therapy wi ...

PMID 25622086.0 | Vemurafenib beyond progression in a patient with metastatic melanoma: a case report. (2015.0)
Abstract: The prognosis of metastatic melanoma has changed markedly in recent years because of the advent of newer targeted therapies such as BRAF inhibitors. However, the response to BRAF inhibitor therapy i